In [1]:
from pyspark.sql import SparkSession

In [2]:
import sys
sys.version

'3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]'

In [3]:
spark = SparkSession.builder \
    .appName("NorthwindKafkaBatch") \
    .config("spark.sql.shuffle.partitions", "4") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

In [4]:
spark.sparkContext.master

'spark://spark-master:7077'

In [5]:
kafka_df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "northwind_customer_monthly_batch") \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load()

In [6]:
kafka_df

DataFrame[key: binary, value: binary, topic: string, partition: int, offset: bigint, timestamp: timestamp, timestampType: int]

In [7]:
# Decode Kafka Value (JSON → Columns)
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import *

schema = StructType([
    StructField("batch_id", StringType()),
    StructField("row_count", IntegerType()),
    StructField("data", ArrayType(
        StructType([
            StructField("order_month", StringType()),
            StructField("customer_id", StringType()),
            StructField("company_name", StringType()),
            StructField("total_orders", IntegerType()),
            StructField("total_revenue", DoubleType()),
            StructField("published_at", StringType(), True)
        ])
    ))
])

In [8]:
parsed_df = kafka_df.select(
    from_json(col("value").cast("string"), schema).alias("json")
)

In [9]:
# flatten (explode batch data)
final_df = (
    parsed_df
    .select(
        col("json.batch_id"),
        explode(col("json.data")).alias("row")
    )
    .select(
        col("batch_id"),
        col("row.order_month"),
        col("row.customer_id"),
        col("row.company_name"),
        col("row.total_orders"),
        col("row.total_revenue")
    )
)

In [10]:
final_df

DataFrame[batch_id: string, order_month: string, customer_id: string, company_name: string, total_orders: int, total_revenue: double]

In [11]:
hadoop_conf = spark._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "minio")
hadoop_conf.set("fs.s3a.secret.key", "minio123")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

In [12]:
# tulis ke minio
final_df.write \
    .mode("overwrite") \
    .partitionBy("order_month") \
    .parquet("s3a://northwind-batch/spark/northwind_customer_monthly/")

In [13]:
final_df.printSchema()

root
 |-- batch_id: string (nullable = true)
 |-- order_month: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- total_orders: integer (nullable = true)
 |-- total_revenue: double (nullable = true)



In [15]:
spark.read.parquet(
    "s3a://northwind-batch/spark/northwind_customer_monthly/"
).printSchema()

root
 |-- batch_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- total_orders: integer (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- order_month: string (nullable = true)

